# 2. Dataset Preprocessing
Questo notebook prepara i dataset crudi convertendoli nel formato compatibile con YOLO e SAM.

In [1]:
import os, json, shutil, base64, zlib, cv2, numpy as np, random
from pathlib import Path
from tqdm import tqdm
print('✅ Librerie caricate')

✅ Librerie caricate


In [2]:
# ── Setup Cartelle ─────────────────────────────────────────────────────────────
BASE = Path('/teamspace/studios/this_studio/project-sbs')
DATASETS = BASE / 'datasets'
RAW_YOLO = DATASETS / 'dati_grezzi_yolo'
RAW_SAM  = DATASETS / 'dati_grezzi_sam'
PROC_YOLO= DATASETS / 'dataset_yolo'
PROC_SAM = DATASETS / 'dataset_sam'
ETICHETTE= RAW_YOLO / 'etichette_yolo_temp'

for d in [PROC_YOLO, PROC_SAM, ETICHETTE]:
    d.mkdir(parents=True, exist_ok=True)

In [3]:
# ── Preprocessing YOLO (JSON to TXT) ────────────────────────────────────────
def get_banner_boxes(json_path):
    data = json.loads(json_path.read_text(encoding='utf-8'))
    iw, ih = data['size']['width'], data['size']['height']
    objects = data.get('objects', [])
    if not objects: return None, iw, ih
    raw_boxes = []
    for obj in objects:
        if obj.get('geometryType') != 'bitmap': continue
        bm = obj.get('bitmap', {})
        if 'data' not in bm or 'origin' not in bm: continue
        try:
            raw = zlib.decompress(base64.b64decode(bm['data']))
            arr = np.frombuffer(raw, np.uint8)
            img = cv2.imdecode(arr, cv2.IMREAD_UNCHANGED)
            if img is None: continue
            h_bm, w_bm = img.shape[0], img.shape[1]
            ox, oy = bm['origin']
            x1, y1 = max(0, ox), max(0, oy)
            x2, y2 = min(iw-1, ox+w_bm), min(ih-1, oy+h_bm)
            if x2>x1 and y2>y1: raw_boxes.append([x1,y1,x2,y2])
        except: continue
    if not raw_boxes: return None, iw, ih
    bands = []
    for box in sorted(raw_boxes, key=lambda b: (b[1]+b[3])/2):
        cy = (box[1]+box[3])/2
        added = False
        for band in bands:
            cy_band = (band[1]+band[3])/2
            if abs(cy-cy_band) < 0.12*ih:
                band[0] = min(band[0], box[0]); band[1] = min(band[1], box[1])
                band[2] = max(band[2], box[2]); band[3] = max(band[3], box[3])
                added = True; break
        if not added: bands.append(list(box))
    final = []
    for x1,y1,x2,y2 in bands:
        bw, bh = x2-x1, y2-y1
        pad_x, pad_y = int(bw*0.12), int(max(4, bh*0.20))
        x1, x2 = max(0, x1-pad_x), min(iw-1, x2+pad_x)
        y1, y2 = max(0, y1-pad_y), min(ih-1, y2+pad_y)
        final.append([x1,y1,x2,y2])
    return final, iw, ih

def to_yolo(box, iw, ih):
    x1, y1, x2, y2 = box
    bw, bh = x2-x1, y2-y1
    if bw < 2 or bh < 2: return None
    cx, cy, nw, nh = (x1+x2)/2/iw, (y1+y2)/2/ih, bw/iw, bh/ih
    if nw < 0.005 or nh < 0.005: return None
    return f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'

In [4]:
ann_dir = RAW_YOLO / 'football' / 'annotations'
img_dir = RAW_YOLO / 'football' / 'images'
if list(ETICHETTE.glob('*.txt')):
    print('✅ JSON -> YOLO conversion already done')
else:
    all_jsons = sorted(ann_dir.glob('*.json'))
    for jf in tqdm(all_jsons, desc='Conv YOLO'):
        stem = jf.name.replace('.png.json','').replace('.jpg.json','').replace('.json','')
        imgp = img_dir / f'{stem}.png'
        if not imgp.exists(): imgp = img_dir / f'{stem}.jpg'
        if not imgp.exists(): continue
        boxes, iw, ih = get_banner_boxes(jf)
        if not boxes: continue
        lines = [to_yolo(b, iw, ih) for b in boxes]
        lines = [l for l in lines if l is not None]
        if lines:
            (ETICHETTE / f'{stem}.txt').write_text('\n'.join(lines), encoding='utf-8')

Conv YOLO: 100%|██████████| 8936/8936 [00:09<00:00, 906.56it/s] 


In [5]:
# ── Train/Val/Test Split YOLO ──────────────────────────────────────────────
for s in ['train', 'val', 'test']:
    (PROC_YOLO / 'images' / s).mkdir(parents=True, exist_ok=True)
    (PROC_YOLO / 'labels' / s).mkdir(parents=True, exist_ok=True)
if list((PROC_YOLO / 'images' / 'train').glob('*.*')):
    print('✅ Split YOLO already done')
else:
    pairs = []
    for lbl in sorted(ETICHETTE.glob('*.txt')):
        stem = lbl.stem
        imgp = img_dir / f'{stem}.png'
        if not imgp.exists(): imgp = img_dir / f'{stem}.jpg'
        if imgp.exists(): pairs.append((imgp, lbl))
    random.seed(42)
    random.shuffle(pairs)
    n = len(pairs)
    splits = {
        'train': pairs[:int(n*0.8)],
        'val': pairs[int(n*0.8):int(n*0.9)],
        'test': pairs[int(n*0.9):]
    }
    for sname, spairs in splits.items():
        for imgp, lblp in spairs:
            shutil.copy(imgp, PROC_YOLO / 'images' / sname / imgp.name)
            shutil.copy(lblp, PROC_YOLO / 'labels' / sname / lblp.name)
    print(f'✅ YOLO splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')
    shutil.rmtree(ETICHETTE)

✅ YOLO splits: train=7063, val=883, test=883


In [6]:
# ── Preprocessing SAM (Mask to Polygon JSON) ─────────────────────────────────
mask_dir = RAW_SAM / 'Masks' / 'Masks'
img_sam_dir  = RAW_SAM / 'Tagged_Images' / 'Tagged Images'
if list(PROC_SAM.rglob('*.json')):
    print('✅ Preprocess & Split SAM già fatto. Skip.')
else:
    def mask_to_poly(mask_path):
        mask = cv2.imread(str(mask_path), 0)
        if mask is None: return []
        mask = (mask > 127).astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return [c.flatten().tolist() for c in contours if cv2.contourArea(c) > 100]
    all_masks = sorted(mask_dir.glob('mask*.png'))
    stems_sam = []
    for maskp in tqdm(all_masks, desc='Prep SAM'):
        idx  = maskp.stem.replace('mask', '')
        imgp = img_sam_dir / f'frame{idx}.jpg'
        if not imgp.exists(): continue
        polys = mask_to_poly(maskp)
        if not polys: continue
        ann = { 'filename': imgp.name, 'regions': [{'all_points_x': polys[0][0::2], 'all_points_y': polys[0][1::2]}] }
        stem_path = PROC_SAM / f'{idx}.json'
        stem_path.write_text(json.dumps(ann))
        shutil.copy(imgp, PROC_SAM / imgp.name)
        stems_sam.append(idx)
    random.seed(42)
    random.shuffle(stems_sam)
    splits_sam = {'train': stems_sam[:int(0.8*len(stems_sam))], 'val': stems_sam[int(0.8*len(stems_sam)):]}
    for split, sstems in splits_sam.items():
        sdir = PROC_SAM / split
        sdir.mkdir(parents=True, exist_ok=True)
        for stem in sstems:
            for ext in ['.jpg', '.png']:
                src = PROC_SAM / f'frame{stem}{ext}'
                if src.exists():
                    shutil.copy(src, sdir / src.name)
                    os.remove(src)
            jsrc = PROC_SAM / f'{stem}.json'
            if jsrc.exists():
                shutil.copy(jsrc, sdir / jsrc.name)
                os.remove(jsrc)
    print('✅ SAM preprocessing completato')

Prep SAM: 100%|██████████| 1620/1620 [00:22<00:00, 71.33it/s]


✅ SAM preprocessing completato
